# 12 - Synthetic Benchmark Validation Framework Summary

This notebook is intentionally table-driven. It reads the artifacts produced by
`scripts/validate_synthetic_benchmark.py` and `scripts/build_benchmark_registries.py`
and presents release readiness, warnings, and discrepancy details without
reimplementing validation logic in notebook cells.

Read the statuses with the specification's decision rule in mind:

- **PASS** — the interval lies inside the predeclared tolerance, or the invariant
  holds exactly. For metrics carrying no bootstrap interval this means "not
  obviously discrepant", which is weaker than equivalence.
- **WARNING** — the estimate crosses a tolerance, the evidence is too thin to
  decide, or a non-critical discrepancy remains.
- **FAIL** — a critical invariant broke, truth leaked, a real record was copied,
  or a core benchmark property is materially outside tolerance.
- **INCONCLUSIVE** — the check could not be run on the available data. Never read
  it as a pass.

Do not average these into one score: a high average hides a fatal defect.


In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VERSION = "v0_3_temporal_candidate_revision"
BASE = PROJECT_ROOT / "reports/tables/synthetic_benchmark" / VERSION
VALIDATION_DIR = BASE / "validation_framework"
REGISTRY_DIR = BASE / "registries"

metrics = pd.read_csv(VALIDATION_DIR / "validation_metrics_long.csv")
gates = pd.read_csv(VALIDATION_DIR / "validation_gate_summary.csv")
discrepancies = pd.read_csv(VALIDATION_DIR / "discrepancy_register.csv")
probes = pd.read_csv(VALIDATION_DIR / "probe_linker_results.csv")
manifest = json.loads((VALIDATION_DIR / "validation_manifest.json").read_text())

{k: v for k, v in manifest.items() if k not in {"input_checksums", "probe_linker_results"}}

{'benchmark_version': 'v0_3_temporal_candidate_revision',
 'scenario': 'central_provisional',
 'world': '001',
 'corruption': '001',
 'synthetic_dir': 'data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisional/world_001/corruption_001',
 'generated_at_utc': '2026-07-28T08:23:05.005032+00:00',
 'strict_60m': False,
 'bootstrap_reps': 200,
 'robustness_sweep': True,
 'canonical_replay_checked': True,
 'probe_replicate_results': [{'scenario': 'central_provisional',
   'world': '001',
   'corruption': '001',
   'probe': 'deterministic_top1',
   'n_predicted_pairs': 94,
   'pair_precision': 0.35106382978723405,
   'pair_recall': 0.275,
   'pair_f1': 0.30841121495327106,
   'bcubed_precision': 0.8508688783570298,
   'bcubed_recall': 0.7890995260663505,
   'bcubed_f1': 0.8188209319705932},
  {'scenario': 'central_provisional',
   'world': '001',
   'corruption': '001',
   'probe': 'score_threshold_all_ranks',
   'n_predicted_pairs': 39,
   'pair_precision': 0.717

## Release gate

Critical gates block release on their own. Non-critical gates can only downgrade
the run to `PASS_WITH_WARNINGS`, and their breaches belong in the fidelity budget:
the benchmark is allowed to simplify the real corpus as long as the simplification
is written down.

In [2]:
gates.sort_values(["critical", "status", "gate"], ascending=[False, True, True])

,gate,status,critical,n_pass,n_warning,n_fail,n_inconclusive,headline,blocking_reason,warning_reason
13,algorithm_utility,PASS,True,34,0,0,0,algorithm_utility validation PASS,NaN,NaN
5,candidate_environment,PASS,True,10,0,0,0,candidate_environment validation PASS,NaN,NaN
0,internal,PASS,True,35,0,0,0,internal validation PASS,NaN,NaN
11,privacy,PASS,True,4,0,0,0,privacy validation PASS,NaN,NaN
1,specification_recovery,PASS,True,6,0,0,0,specification_recovery validation PASS,NaN,NaN
3,conditionals,WARNING,True,3,1,0,0,conditionals validation WARNING,NaN,siret_present:WMAE_pp
12,hidden_truth_difficulty,WARNING,True,10,2,0,0,hidden_truth_difficulty validation WARNING,NaN,pairs_quality:PQ; pairs_quality:PQ
4,temporal,WARNING,True,8,1,0,0,temporal validation WARNING,NaN,followup_runway:abs_diff_pp
8,missingness_structure,PASS,False,4,0,0,0,missingness_structure validation PASS,NaN,NaN
7,buyer_activity,WARNING,False,7,5,0,0,buyer_activity validation WARNING,NaN,activity_gini:abs_diff; activity_share:abs_diff_pp; activity_share:abs_diff_pp; activi...


### What is currently blocking

In [3]:
blocking = gates[gates["critical"] & gates["status"].eq("FAIL")]
if blocking.empty:
    print("No critical gate is failing.")
else:
    display(blocking[["gate", "n_fail", "blocking_reason"]])
    display(
        metrics[metrics["scope"].isin(blocking["gate"]) & metrics["status"].eq("FAIL")][
            ["property", "metric", "synthetic_estimate", "ci_low", "ci_high", "tolerance", "notes"]
        ]
    )

No critical gate is failing.


### Metric failures carried under non-critical gates

A non-critical gate reports `WARNING` even when individual metrics fail, so the
headline status alone understates how many metric-level failures the release is
carrying. These do not block under the documented gate policy, but they are the
fidelity debt the benchmark is shipping with.

In [4]:
print(
    manifest["n_metric_failures"],
    "metric failures total;",
    manifest["n_metric_failures_in_noncritical_gates"],
    "inside non-critical gates:",
    manifest["noncritical_gates_with_metric_failures"],
)
carried = gates[(~gates["critical"]) & (gates["n_fail"] > 0)]
display(carried[["gate", "status", "n_pass", "n_fail", "warning_reason"]])

2 metric failures total; 2 inside non-critical gates: ['missingness_text_identifier']


,gate,status,n_pass,n_fail,warning_reason
6,missingness_text_identifier,WARNING,6,2,siret_missing:rate_abs_diff_pp; siret_present:presence_rate_abs_diff_pp; siren_missing...


### Reproducibility

`canonical_replay_matches_released_tables` regenerates the benchmark from the
recorded seeds and the live scenario file and compares canonical table content.
`scenario_snapshot_matches_scenario_file` checks that the scenario snapshot
saved with the run still matches that file, so configuration drift is named
rather than surfacing as an unexplained hash mismatch.

In [5]:
print("canonical replay checked:", manifest["canonical_replay_checked"])
metrics[metrics["property"].eq("reproducibility")][["metric", "status", "notes"]]

canonical replay checked: True


,metric,status,notes
31,manifest_records_seeds_and_code_version,PASS,missing=[]
32,scenario_snapshot_matches_scenario_file,PASS,recorded scenario snapshot matches the live scenario file
33,corruption_log_seed_matches_manifest,PASS,declared=20260722; logged=[20260722]
34,canonical_replay_matches_released_tables,PASS,9/9 tables reproduced from the recorded seeds and the live scenario file


## Discrepancy register

In [6]:
discrepancies.sort_values(["status", "scope", "property", "metric"])

,scope,subgroup,property,metric,real_estimate,synthetic_estimate,difference,effect_size,tolerance,status,provenance,notes
3,missingness_text_identifier,overall,siret_missing,rate_abs_diff_pp,0.727757,0.684026,-4.373142,4.373142,2.00,FAIL,observable_real_vs_synthetic,NaN
5,missingness_text_identifier,overall,siret_present,presence_rate_abs_diff_pp,0.272243,0.315974,4.373142,4.373142,2.00,FAIL,observable_real_vs_synthetic,NaN
12,names_identifiers,overall,siren_invalid_among_present,abs_diff_pp,NaN,0.000000,NaN,NaN,2.00,INCONCLUSIVE,observable_real_vs_synthetic,present raw identifier that fails format/checksum validation
7,buyer_activity,overall,activity_gini,abs_diff,0.831614,0.713858,-0.117756,0.117756,0.10,WARNING,observable_real_vs_synthetic,bootstrap_reps=200; n_real_buyers=5268; n_syn_buyers=1158
8,buyer_activity,top1pct,activity_share,abs_diff_pp,0.382780,0.306464,-7.631570,7.631570,10.00,WARNING,observable_real_vs_synthetic,share of all notices held by the top1pct most active buyers
9,buyer_activity,top5pct,activity_share,abs_diff_pp,0.661286,0.541945,-11.934096,11.934096,10.00,WARNING,observable_real_vs_synthetic,share of all notices held by the top5pct most active buyers
10,buyer_activity,top10pct,activity_share,abs_diff_pp,0.779823,0.655949,-12.387417,12.387417,10.00,WARNING,observable_real_vs_synthetic,share of all notices held by the top10pct most active buyers
11,buyer_activity,overall,relative_notices_per_buyer,q99_abs_diff,16.205591,14.365860,-1.839732,1.839732,1.00,WARNING,observable_real_vs_synthetic,activity divided by that corpus's own mean activity (scale-free)
1,conditionals,"schema_family,publication_year,notice_type_normalized",siret_present,WMAE_pp,NaN,NaN,5.056234,5.056234,3.00,WARNING,observable_real_vs_synthetic,eligible_cells=30; total_cells=45
15,hidden_truth_difficulty,production_window,pairs_quality,PQ,NaN,0.038435,NaN,0.038435,0.05,WARNING,synthetic_truth,56 true matches among 1457 candidate pairs


## Metric inventory by gate

In [7]:
(
    metrics.groupby(["scope", "status"])
    .size()
    .rename("n")
    .reset_index()
    .pivot(index="scope", columns="status", values="n")
    .fillna(0)
    .astype(int)
)

status,FAIL,INCONCLUSIVE,PASS,WARNING
scope,,,,
algorithm_utility,0,0,34,0
buyer_activity,0,0,7,5
candidate_environment,0,0,10,0
conditionals,0,0,3,1
hidden_truth_difficulty,0,0,10,2
internal,0,0,35,0
marginals,0,0,17,1
missingness_structure,0,0,4,0
missingness_text_identifier,2,0,6,2


## Benchmark difficulty

These need the sealed truth tables and cannot be computed on real BOAMP at all.
Blocking recall says whether a comparison model is even being given the chance to
work; score overlap says whether telling a match from a plausible non-match takes
real discrimination.

In [8]:
metrics[metrics["scope"].eq("hidden_truth_difficulty")][
    ["subgroup", "property", "metric", "synthetic_estimate", "tolerance", "status"]
]

,subgroup,property,metric,synthetic_estimate,tolerance,status
132,algorithm_scope,true_matches_in_scope,count,120,context_only,PASS
133,production_window,blocking_pairs_completeness,PC,0.4666666666666667,0.25,PASS
134,PRODUCTION_BLOCKING,evaluation_setting,R_blocking,0.4666666666666667,0.25,PASS
135,production_window,pairs_quality,PQ,0.038435140700068635,0.05,WARNING
136,production_window,reduction_ratio,RR,0.9835980682419425,0.95,PASS
137,36m_window,blocking_pairs_completeness,PC,0.9416666666666667,0.8,PASS
138,36M_BLOCKING,evaluation_setting,R_blocking,0.9416666666666667,0.8,PASS
139,36m_window,pairs_quality,PQ,0.030213903743315507,0.05,WARNING
140,36m_window,reduction_ratio,RR,0.9578975807994957,0.95,PASS
141,production_window,match_vs_hard_negative_score,overlap,0.3406240440501683,0.05-0.95,PASS


## Probe linkers

Three deliberately different linkers with frozen parameters. None of them is the
production acceptance threshold — a pipeline threshold is an output of that
pipeline, never a definition of truth. The benchmark is informative if the probes
separate, and unusable if any of them solves it.

In [9]:
probes

,probe,n_predicted_pairs,pair_precision,pair_recall,pair_f1,bcubed_precision,bcubed_recall,bcubed_f1
0,deterministic_top1,94,0.351064,0.275000,0.308411,0.850869,0.789100,0.818821
1,score_threshold_all_ranks,39,0.717949,0.233333,0.352201,0.977488,0.775276,0.864718
2,fellegi_sunter,151,0.172185,0.216667,0.191882,0.732510,0.781991,0.756442


## Predeclared tolerances and parameter provenance

Tolerances are read from the gate code itself, so a retuned tolerance shows up here
instead of quietly diverging from the registry. Parameters marked
`SCENARIO_UNIDENTIFIED` encode assumptions real BOAMP cannot settle and must be
varied across scenarios rather than quoted as estimates.

In [10]:
tolerances = pd.read_csv(REGISTRY_DIR / "tolerance_registry.csv")
parameters = pd.read_csv(REGISTRY_DIR / "parameter_registry.csv")

display(tolerances[tolerances["gate_is_critical"]][["module", "tolerance_key", "value", "gate"]].drop_duplicates())
parameters["provenance_class"].value_counts()

,module,tolerance_key,value,gate
1,fidelity,categorical_tv,0.05,conditionals
2,fidelity,categorical_tv,0.05,temporal
3,fidelity,categorical_tv,0.05,candidate_environment
6,fidelity,categorical_js,0.05,conditionals
7,fidelity,categorical_js,0.05,temporal
...,...,...,...,...
194,difficulty,probe_best_f1_max,0.99,algorithm_utility
195,difficulty,probe_best_f1_min,0.05,hidden_truth_difficulty
196,difficulty,probe_best_f1_min,0.05,algorithm_utility
197,difficulty,probe_f1_spread_min,0.01,hidden_truth_difficulty


provenance_class
EMPIRICAL_OBSERVABLE             263
SCENARIO_UNIDENTIFIED            206
ALGORITHM_PARAMETER               77
SILVER_STANDARD_APPROXIMATION     28
IMPLEMENTATION_CONSTANT           14
FIDELITY_TARGET                    5
Name: count, dtype: int64

### Parameters whose scenario-file value was overridden at run time

In [11]:
parameters[parameters["overridden_at_runtime"]][
    ["parameter", "scenario_file_value", "effective_value", "provenance_class"]
]

,parameter,scenario_file_value,effective_value,provenance_class


## Replicate inventory

Seed-to-seed stability cannot be estimated from a single world per scenario, so it
is reported `INCONCLUSIVE` rather than assumed.

In [12]:
display(pd.read_csv(REGISTRY_DIR / "scenario_manifest.csv"))
metrics[metrics["scope"].eq("robustness")][["subgroup", "property", "metric", "synthetic_estimate", "status", "notes"]]

,benchmark_version,scenario,manifest_row_type,config_status,artifact_status,world,corruption,world_seed,corruption_seed,generator_version,git_commit,n_observed_notices,replicate_index_for_scenario,generated_replicates_for_scenario,path
0,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,1.0,1.0,20260721.0,20260722.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,8833.0,1.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
1,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,2.0,2.0,20260731.0,20260732.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,9261.0,2.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
2,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,3.0,3.0,20260741.0,20260742.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,8607.0,3.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
3,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,4.0,4.0,20260751.0,20260752.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,9060.0,4.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
4,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,5.0,5.0,20260761.0,20260762.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,8649.0,5.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
5,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,6.0,6.0,20260771.0,20260772.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,9048.0,6.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
6,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,7.0,7.0,20260781.0,20260782.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,9370.0,7.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
7,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,8.0,8.0,20260791.0,20260792.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,9379.0,8.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
8,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,9.0,9.0,20260801.0,20260802.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,9091.0,9.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
9,v0_3_temporal_candidate_revision,central_provisional,GENERATED_REPLICATE,CONFIGURED,GENERATED,10.0,10.0,20260811.0,20260812.0,0.3.0-temporal-candidate-revision,30da6f5532a05a91faebf6e48aa2aba9b1ff74d5,8737.0,10.0,10,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...


,subgroup,property,metric,synthetic_estimate,status,notes
178,all_replicates,replicate_inventory,count,51,PASS,"scenarios=['central_provisional', 'clean_sanity', 'difficult', 'easier', 'moderate', '..."
179,seed_replicates:central_provisional,blocking_pairs_completeness,seed_count,10,PASS,number of generated seed replicates included in the within-scenario summary
180,seed_replicates:central_provisional,blocking_pairs_completeness,mean,0.3994205424273881,PASS,within-scenario mean over 10 generated seeds
181,seed_replicates:central_provisional,blocking_pairs_completeness,std,0.12842299829793574,PASS,within-scenario standard deviation over 10 generated seeds
182,seed_replicates:central_provisional,blocking_pairs_completeness,central_95pct_interval,0.18189-0.596143,PASS,empirical 2.5%-97.5% interval across generated seeds
183,seed_replicates:central_provisional,blocking_pairs_completeness,worst_case,0.1449067431850789,PASS,minimum value observed across generated seeds
184,seed_replicates:central_provisional,blocking_pairs_completeness,coefficient_of_variation,0.32152326847656354,WARNING,within-scenario seed stability; high values mean one seed is not representative
185,seed_replicates:central_provisional,blocking_pairs_completeness,nonpass_frequency,0.1,WARNING,share of generated seeds whose underlying headline metric was not PASS (WARNING/FAIL/I...
186,seed_replicates:central_provisional,match_vs_hard_negative_score,seed_count,10,PASS,number of generated seed replicates included in the within-scenario summary
187,seed_replicates:central_provisional,match_vs_hard_negative_score,mean,0.30719817307981173,PASS,within-scenario mean over 10 generated seeds
